# 0.0 Set up

In [1]:
import re
import os, sys
import importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path 
import geopandas as gpd

# Add project root to Python path
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
  sys.path.insert(0, str(project_root))

# Import config and setup
from config import setup_notebook, get_path
setup_notebook()

✓ Project root: /workspace/project_paaral
✓ Working directory: /workspace/project_paaral
✓ Python path updated


{'project_root': PosixPath('/workspace/project_paaral'),
 'data': PosixPath('/workspace/project_paaral/data'),
 'modules': PosixPath('/workspace/project_paaral/modules'),
 'notebooks': PosixPath('/workspace/project_paaral/notebooks'),
 'output': PosixPath('/workspace/project_paaral/output'),
 'psgc_shapefiles': PosixPath('/workspace/project_paaral/data/philippines-psgc-shapefiles/dist')}

# 1.0 Map resources

## 1.1 PSGC shapefiles

In [2]:
from modules import psgc_consolidator

# Reload the module to get latest changes
importlib.reload(psgc_consolidator)

from modules import psgc_consolidator as pc

In [3]:
# Initialize consolidator
path = get_path('psgc_shapefiles') # "data/philippines-psgc-shapefiles/dist"
consolidator = pc.PSGCConsolidator(base_path=path, verbose=False)

# Process data
geodata = consolidator.process()

# # Get summary
# summary = consolidator.get_summary()

/usr/local/lib/python3.11/site-packages/pyogrio/geopandas.py:275: UserWarning: More than one layer found in 'PH_Adm4_BgySubMuns.shp.zip': 'BgySubMuns.shp' (default), 'PH_Adm4_BgySubMuns.shp'. Specify layer parameter to avoid this warning.
  result = read_func(


In [4]:
print(geodata.shape)
with pd.option_context('display.max_columns', None):
    display(geodata.head())

(42048, 18)


,psgc_code,corr_code,name,adm4_en_psgc,geometry,adm1_psgc,adm2_psgc,adm3_psgc,adm4_psgc,adm1_en,adm2_en,adm3_en,adm4_en_shapes,geo_level,len_crs,area_crs,len_km,area_km2
0,0102801001,12801001.0,Adams,Adams (Pob.),"POLYGON ((120.92068 18.51462, 120.94626 18.511...",0100000000,0102800000,0102801000,0102801001,Region I (Ilocos Region),Ilocos Norte,Adams,Adams,Bgy,45997.0,111184551.0,45.0,111.0
1,0102802001,12802001.0,Bani,Bani,"POLYGON ((120.61278 18.2759, 120.61282 18.2758...",0100000000,0102800000,0102802000,0102802001,Region I (Ilocos Region),Ilocos Norte,Bacarra,Bani,Bgy,5982.0,1761135.0,5.0,1.0
2,0102802002,12802002.0,Buyon,Buyon,"POLYGON ((120.62741 18.24638, 120.6337 18.2413...",0100000000,0102800000,0102802000,0102802002,Region I (Ilocos Region),Ilocos Norte,Bacarra,Buyon,Bgy,9117.0,3875134.0,9.0,3.0
3,0102802003,12802003.0,Cabaruan,Cabaruan,"POLYGON ((120.58982 18.26839, 120.60407 18.259...",0100000000,0102800000,0102802000,0102802003,Region I (Ilocos Region),Ilocos Norte,Bacarra,Cabaruan,Bgy,7745.0,2987648.0,7.0,2.0
4,0102802004,12802004.0,Cabulalaan,Cabulalaan,"POLYGON ((120.58368 18.2821, 120.58401 18.2820...",0100000000,0102800000,0102802000,0102802004,Region I (Ilocos Region),Ilocos Norte,Bacarra,Cabulalaan,Bgy,4502.0,1018354.0,4.0,1.0


In [5]:
geodata.isna().sum()

psgc_code            0
corr_code         3591
name              3582
adm4_en_psgc         0
geometry             0
adm1_psgc         3580
adm2_psgc         3580
adm3_psgc         3580
adm4_psgc         3580
adm1_en           3580
adm2_en           3580
adm3_en           3582
adm4_en_shapes    3582
geo_level         3582
len_crs           3580
area_crs          3580
len_km            3580
area_km2          3580
dtype: int64

In [ ]:
shapes_adm4 = consolidator.adm4_geodata.copy()
print(shapes_adm4.shape)
display(shapes_adm4.head(3))

In [ ]:
psgc_consolidated = consolidator.consolidated_data.copy()
print(psgc_consolidated.shape)
display(psgc_consolidated.head())

In [ ]:
psgc_ncr = psgc_consolidated[psgc_consolidated['adm1_en'].astype('string').str.contains(r'capital', flags=re.IGNORECASE)]
psgc_ncr_cities = psgc_ncr['adm3_en'].unique()
display(psgc_ncr_cities)

# City of Manila is the only missing city in the unique values of adm3_en
mask = (
    (psgc_ncr['adm3_en'].isna())
    & (psgc_ncr['adm2_en'].isna())
)
psgc_ncr_manila = psgc_ncr.loc[mask].copy()
print(psgc_ncr_manila.shape)
display(psgc_ncr_manila.head())

In [ ]:
!ls "data/philippines-psgc-shapefiles/dist/PH_Adm4_BgySubMuns.shp.zip"

In [ ]:
shapefile_path = "data/philippines-psgc-shapefiles/dist/PH_Adm4_BgySubMuns.shp.zip"
adm4_shapefile = gpd.read_file(shapefile_path)

In [ ]:
adm4_shapefile

### 1.1.1 Plotting helpers

In [ ]:
from matplotlib.patches import Rectangle
import warnings
warnings.filterwarnings('ignore')

def plot_philippines_shapefile(gdf, 
                              title="Philippines Map",
                              figsize=(12, 14),
                              color='lightblue',
                              edgecolor='black',
                              linewidth=0.5,
                              simplified=True,
                              simplify_tolerance=0.01,
                              use_bounds=True,
                              dpi=100):
    """
    Efficiently plot Philippines shapefile from GeoDataFrame
    
    Parameters:
    -----------
    gdf : GeoDataFrame
        GeoDataFrame containing Philippines geometry
    title : str
        Title for the plot
    figsize : tuple
        Figure size (width, height)
    color : str
        Fill color for polygons
    edgecolor : str
        Edge color for polygons
    linewidth : float
        Width of polygon edges
    simplified : bool
        Whether to simplify geometry for faster rendering
    simplify_tolerance : float
        Tolerance for simplification (higher = more simplified)
    use_bounds : bool
        Whether to set specific bounds for Philippines
    dpi : int
        DPI for the figure (lower = faster rendering)
    
    Returns:
    --------
    fig, ax : matplotlib figure and axes objects
    """
    
    # Create figure with lower DPI for faster rendering
    fig, ax = plt.subplots(1, 1, figsize=figsize, dpi=dpi)
    
    # Simplify geometry if requested (reduces computational load)
    if simplified:
        gdf_plot = gdf.copy()
        gdf_plot['geometry'] = gdf_plot['geometry'].simplify(
            tolerance=simplify_tolerance, 
            preserve_topology=True
        )
    else:
        gdf_plot = gdf
    
    # Plot with minimal styling for efficiency
    gdf_plot.plot(
        ax=ax,
        color=color,
        edgecolor=edgecolor,
        linewidth=linewidth,
        alpha=0.8
    )
    
    # Set bounds specifically for Philippines to avoid unnecessary rendering
    if use_bounds:
        # Philippines approximate bounds
        ax.set_xlim([116.5, 127.5])  # Longitude
        ax.set_ylim([4.5, 21.5])     # Latitude
    
    # Minimal styling for efficiency
    ax.set_title(title, fontsize=14, fontweight='bold', pad=20)
    ax.set_xlabel('Longitude', fontsize=10)
    ax.set_ylabel('Latitude', fontsize=10)
    
    # Add simple grid
    ax.grid(True, alpha=0.3, linestyle='--', linewidth=0.5)
    
    # Remove top and right spines for cleaner look
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    # Aspect ratio suitable for Philippines
    ax.set_aspect('equal')
    
    plt.tight_layout()
    
    return fig, ax


def plot_philippines_regions(gdf, 
                            region_column=None,
                            cmap='Set3',
                            show_labels=False,
                            label_column=None,
                            **kwargs):
    """
    Plot Philippines with regions in different colors
    
    Parameters:
    -----------
    gdf : GeoDataFrame
        GeoDataFrame containing Philippines geometry
    region_column : str
        Column name containing region information
    cmap : str
        Colormap for regions
    show_labels : bool
        Whether to show region labels
    label_column : str
        Column to use for labels (if different from region_column)
    **kwargs : additional arguments passed to plot_philippines_shapefile
    
    Returns:
    --------
    fig, ax : matplotlib figure and axes objects
    """
    
    # Set default figure size if not provided
    if 'figsize' not in kwargs:
        kwargs['figsize'] = (12, 14)
    
    # Create figure
    fig, ax = plt.subplots(1, 1, figsize=kwargs.get('figsize'), 
                          dpi=kwargs.get('dpi', 100))
    
    # Simplify if requested
    if kwargs.get('simplified', True):
        gdf_plot = gdf.copy()
        gdf_plot['geometry'] = gdf_plot['geometry'].simplify(
            tolerance=kwargs.get('simplify_tolerance', 0.01),
            preserve_topology=True
        )
    else:
        gdf_plot = gdf
    
    # Plot with regions
    if region_column and region_column in gdf_plot.columns:
        gdf_plot.plot(
            column=region_column,
            ax=ax,
            cmap=cmap,
            edgecolor=kwargs.get('edgecolor', 'black'),
            linewidth=kwargs.get('linewidth', 0.5),
            alpha=0.8,
            legend=True,
            legend_kwds={'loc': 'upper left', 'fontsize': 8}
        )
    else:
        gdf_plot.plot(
            ax=ax,
            color=kwargs.get('color', 'lightblue'),
            edgecolor=kwargs.get('edgecolor', 'black'),
            linewidth=kwargs.get('linewidth', 0.5),
            alpha=0.8
        )
    
    # Add labels if requested (can be computationally expensive for many features)
    if show_labels and label_column and label_column in gdf_plot.columns:
        # Use representative points for label placement
        gdf_plot['label_point'] = gdf_plot.geometry.representative_point()
        
        for idx, row in gdf_plot.iterrows():
            ax.annotate(
                text=row[label_column],
                xy=(row.label_point.x, row.label_point.y),
                horizontalalignment='center',
                fontsize=6,
                fontweight='light'
            )
    
    # Set bounds for Philippines
    if kwargs.get('use_bounds', True):
        ax.set_xlim([116.5, 127.5])
        ax.set_ylim([4.5, 21.5])
    
    # Styling
    ax.set_title(kwargs.get('title', 'Philippines Regions Map'), 
                fontsize=14, fontweight='bold', pad=20)
    ax.set_xlabel('Longitude', fontsize=10)
    ax.set_ylabel('Latitude', fontsize=10)
    ax.grid(True, alpha=0.3, linestyle='--', linewidth=0.5)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.set_aspect('equal')
    
    plt.tight_layout()
    
    return fig, ax


def plot_philippines_minimal(gdf):
    """
    Ultra-minimal plotting for maximum speed
    
    Parameters:
    -----------
    gdf : GeoDataFrame
        GeoDataFrame containing Philippines geometry
    
    Returns:
    --------
    fig, ax : matplotlib figure and axes objects
    """
    
    # Simplify geometry aggressively
    gdf_simplified = gdf.copy()
    gdf_simplified['geometry'] = gdf_simplified['geometry'].simplify(
        tolerance=0.05,  # Higher tolerance for more simplification
        preserve_topology=True
    )
    
    # Plot with minimal settings
    fig, ax = plt.subplots(figsize=(8, 10), dpi=72)  # Lower DPI
    
    gdf_simplified.plot(
        ax=ax,
        color='#E8E8E8',  # Light gray
        edgecolor='#333333',  # Dark gray
        linewidth=0.3
    )
    
    # Minimal decorations
    ax.set_xlim([116.5, 127.5])
    ax.set_ylim([4.5, 21.5])
    ax.set_aspect('equal')
    ax.axis('off')  # Turn off axis for cleanest look
    
    plt.tight_layout()
    
    return fig, ax


# Example usage
if __name__ == "__main__":
    # Example 1: Load shapefile and plot basic map
    # gdf = gpd.read_file("path/to/philippines.shp")
    # fig, ax = plot_philippines_shapefile(gdf)
    # plt.show()
    
    # Example 2: Plot with regions
    # fig, ax = plot_philippines_regions(
    #     gdf, 
    #     region_column='REGION',
    #     show_labels=True,
    #     label_column='NAME_1'
    # )
    # plt.show()
    
    # Example 3: Ultra-fast minimal plot
    # fig, ax = plot_philippines_minimal(gdf)
    # plt.show()
    
    # Example 4: Custom optimization settings
    # fig, ax = plot_philippines_shapefile(
    #     gdf,
    #     simplified=True,
    #     simplify_tolerance=0.02,  # More aggressive simplification
    #     dpi=72,  # Lower DPI for faster rendering
    #     linewidth=0.3  # Thinner lines for faster drawing
    # )
    # plt.show()
    
    pass

### 1.1.2 Plot

#### Entire PH

In [ ]:
# gdf = consolidator.adm4_geodata.copy()

# # Example 1: Plot entire PH
# fig, ax = plot_philippines_shapefile(gdf)
# plt.show()

#### Inspect NCR

In [ ]:
gdf = consolidator.adm4_geodata.copy()
gdf_ncr = gdf[gdf['adm1_pcode'] == 'PH13']

# Example 3: Ultra-fast minimal plot
fig, ax = plt.subplots(figsize=(8,8))

gdf_ncr.plot(
    ax=ax,
    color='none',
    edgecolor='grey',
    linewidth=.5,
    alpha=0.8
)

plt.show()

### 1.1.3 Preferred merge
This section shows how I would merge the consolidated PSGC information and the information accompanying the shapefiles.

In [ ]:
tmp_psgc = consolidator.consolidated_data.copy()

# City of Manila is the only missing city in the unique values of adm3_en
mask = (
    (tmp_psgc['adm1_en'].astype('string').str.contains(r'capital', flags=re.IGNORECASE))
    & (tmp_psgc['adm3_en'].isna())
    & (tmp_psgc['adm2_en'].isna())
)
tmp_psgc.loc[mask, 'adm3_en'] = 'City of Manila'

# Rearrange columns
columns_psgc = [col for col in tmp_psgc.columns if '_psgc' in str(col)]
columns_en = [col for col in tmp_psgc.columns if '_en' in str(col)]
columns_other = [col for col in tmp_psgc.columns if col not in columns_psgc+columns_en]
tmp_psgc = tmp_psgc[columns_psgc+columns_en[::-1]+columns_other]

# Fix psgc codes whose leading 0s were omitted
def add_leading_zeros(psgc_code):
    code_str = str(psgc_code)
    if len(code_str) == 9:
        return '0'+code_str
    else:
        return code_str

for column in columns_psgc:
    tmp_psgc[column] = tmp_psgc[column].apply(add_leading_zeros)
    tmp_psgc[column] = tmp_psgc[column].astype('string')

print(tmp_psgc.shape)
display(tmp_psgc.head(3))

In [ ]:
# tmp_psgc.dtypes

In [ ]:
tmp_psgc.isna().sum()

In [ ]:
tmp_geodata = consolidator.adm4_geodata.copy()

# Add back leading zeros
tmp_geodata['psgc_code'] = tmp_geodata['psgc_code'].apply(add_leading_zeros)
tmp_geodata['psgc_code'] = tmp_geodata['psgc_code'].astype('string')

# Identify relevant columns to be included in merge
columns_relevant = ['psgc_code','corr_code','name','adm4_en','geometry']
tmp_geodata = tmp_geodata[columns_relevant]

# Drop rows without geometry
mask = tmp_geodata['geometry'].isna()
tmp_geodata = tmp_geodata.loc[~mask]

print(tmp_geodata.shape)
tmp_geodata.head(3)

In [ ]:
tmp_geodata.isna().sum()

In [ ]:
consolidated_geodata = tmp_geodata.merge(
    tmp_psgc,
    left_on='psgc_code', right_on='adm4_psgc',
    how='left',
    suffixes=('_psgc','_shapes')
)

# # Drop rows whose adm1_psgc is NaN
# mask = consolidated_geodata['adm1_psgc'].isna()
# consolidated_geodata = consolidated_geodata.loc[~mask]

# Transform to GeoDataFrame
consolidated_geodata = gpd.GeoDataFrame(consolidated_geodata, geometry='geometry', crs=4326)

print(consolidated_geodata.shape)

#### Ideal consolidated output
The cell block below shows my ideal output for the psgc consolidator .py module.

In [ ]:
display(consolidated_geodata.head(5))
display(consolidated_geodata.dtypes)

In [ ]:
consolidated_geodata.isna().sum()

In [ ]:
# Display rows/geometries that do not have a match in the consolidated .CSVs
consolidated_geodata[consolidated_geodata['adm4_en_shapes'].isna()]

In [ ]:
# Example 1: Plot entire PH
fig, ax = plot_philippines_shapefile(consolidated_geodata, figsize=(12,14))
plt.show()

### 1.1.4 Sample of discrepancies
The cell block below shows that the PSGC code in the .shp.zip is different with the PSGC code found in the consolidated .CSV files.

In [ ]:
tmp_psgc.head(1)

In [ ]:
mask = tmp_psgc['adm4_en'].astype('string').str.contains(r'agapito del|anunas|cutud', flags=re.IGNORECASE)
display(tmp_psgc.loc[mask])

## 1.2 Unmatched Barangays Analysis

This section identifies barangays in the shapefile that don't have matching PSGC codes in the CSV data.

In [6]:
# Load the consolidated geodata from the output file
output_path = get_path('output')
geodata_file = output_path / 'consolidated_geodata.gpkg'

# Check if file exists
if geodata_file.exists():
    print(f"Loading: {geodata_file}")
    consolidated_gdf = gpd.read_file(geodata_file)
    print(f"Shape: {consolidated_gdf.shape}")
else:
    # If file doesn't exist, use the consolidated_geodata from section 1.1.3
    print("Using consolidated_geodata from section 1.1.3")
    consolidated_gdf = consolidated_geodata.copy()

Loading: /workspace/project_paaral/output/consolidated_geodata.gpkg
Shape: (42048, 18)


In [7]:
# Identify unmatched barangays (rows with NaN admin codes)
unmatched = consolidated_gdf[consolidated_gdf['adm1_psgc'].isna()].copy()

print(f"Total barangays: {len(consolidated_gdf)}")
print(f"Matched barangays: {len(consolidated_gdf[consolidated_gdf['adm1_psgc'].notna()])}")
print(f"Unmatched barangays: {len(unmatched)}")
print(f"\nUnmatched percentage: {len(unmatched)/len(consolidated_gdf)*100:.2f}%")

# Display sample of unmatched barangays
print("\n" + "="*80)
print("Sample of unmatched barangays:")
print("="*80)
display(unmatched[['psgc_code', 'name', 'adm4_en_psgc', 'geometry']].head(10))

Total barangays: 42048
Matched barangays: 38468
Unmatched barangays: 3580

Unmatched percentage: 8.51%

Sample of unmatched barangays:


,psgc_code,name,adm4_en_psgc,geometry
7236,0305401001,None,Agapito del Rosario,"MULTIPOLYGON (((120.58988 15.14755, 120.59015 ..."
7237,0305401002,None,Anunas,"MULTIPOLYGON (((120.56012 15.16583, 120.56044 ..."
7238,0305401003,None,Balibago,"MULTIPOLYGON (((120.5709 15.22239, 120.57103 1..."
7239,0305401004,None,Capaya,"MULTIPOLYGON (((120.63596 15.15064, 120.63655 ..."
7240,0305401005,None,Claro M. Recto,"MULTIPOLYGON (((120.59362 15.14862, 120.59376 ..."
7241,0305401006,None,Cuayan,"MULTIPOLYGON (((120.49668 15.15778, 120.50238 ..."
7242,0305401007,None,Cutcut,"MULTIPOLYGON (((120.5725 15.14597, 120.57293 1..."
7243,0305401008,None,Cutud,"MULTIPOLYGON (((120.64047 15.18071, 120.64064 ..."
7244,0305401010,None,Lourdes North West,"MULTIPOLYGON (((120.58591 15.1475, 120.58687 1..."
7245,0305401011,None,Lourdes Sur,"MULTIPOLYGON (((120.59127 15.1443, 120.59138 1..."


# 2.0 Spatial Matching for Unmatched Barangays

This section demonstrates the spatial matching approach to fill missing admin codes for unmatched barangays.

## 2.1 Using the PSGC Consolidator's Spatial Matching

The PSGC Consolidator module now includes spatial matching functionality. This approach:
1. Dissolves matched barangays to municipality-level boundaries
2. Uses STRtree spatial indexing for efficient queries
3. Tests centroid-based point-in-polygon containment
4. Falls back to nearest neighbor for boundary cases

In [3]:
# Method 1: Manual control - Apply spatial matching after processing
path = get_path('psgc_shapefiles')
consolidator_matched = pc.PSGCConsolidator(base_path=path, verbose=True)
consolidated = consolidator_matched.process()

print("\n" + "="*80)
print("BEFORE SPATIAL MATCHING")
print("="*80)
print(f"Unmatched barangays: {consolidated['adm1_psgc'].isna().sum()}")

# Apply spatial matching
matched_gdf = consolidator_matched.apply_spatial_matching(save_original=True)

print("\n" + "="*80)
print("AFTER SPATIAL MATCHING")
print("="*80)
print(f"Unmatched barangays: {matched_gdf['adm1_psgc'].isna().sum()}")
print(f"Spatially matched barangays: {matched_gdf['is_spatially_matched'].sum()}")

# DIAGNOSTIC: Check the reference boundaries
print("\n" + "="*80)
print("DIAGNOSTIC: Reference Boundaries")
print("="*80)
print(f"Shape: {consolidator_matched.reference_boundaries.shape}")
print(f"\nSample of reference boundaries:")
display(consolidator_matched.reference_boundaries.head())
print(f"\nNaN counts in reference boundaries:")
print(consolidator_matched.reference_boundaries[['adm1_en', 'adm2_en', 'adm3_en']].isna().sum())

INFO:modules.psgc_consolidator:Starting PSGC consolidation pipeline
INFO:modules.psgc_consolidator:Loading PSGC data from /workspace/project_paaral/data/philippines-psgc-shapefiles/dist
INFO:modules.psgc_consolidator:Loading Adm1 (Regions) data...
INFO:modules.psgc_consolidator:Trimmed whitespaces from 2 string columns
INFO:modules.psgc_consolidator:Loaded 17 regions
INFO:modules.psgc_consolidator:Loading Adm2 (Provinces/Districts) data...
INFO:modules.psgc_consolidator:Trimmed whitespaces from 2 string columns
INFO:modules.psgc_consolidator:Loaded 88 provinces/districts
INFO:modules.psgc_consolidator:Loading Adm3 (Municipalities/Cities) data...
INFO:modules.psgc_consolidator:Trimmed whitespaces from 2 string columns
INFO:modules.psgc_consolidator:Loaded 1642 municipalities/cities
INFO:modules.psgc_consolidator:Loading Adm4 (Barangays/Sub-Municipalities) data...
INFO:modules.psgc_consolidator:Trimmed whitespaces from 2 string columns
INFO:modules.psgc_consolidator:Loaded 42017 barangay


BEFORE SPATIAL MATCHING
Unmatched barangays: 3580


INFO:modules.psgc_consolidator:Created 1582 municipality reference boundaries
INFO:modules.psgc_consolidator:Populating admin names from authoritative sources...
INFO:modules.psgc_consolidator:Reference boundaries name completeness: adm1_en: 1582/1582, adm2_en: 1582/1582, adm3_en: 1581/1582
INFO:modules.psgc_consolidator:Found 3580 unmatched barangays to process
INFO:modules.psgc_consolidator:Starting spatial matching for 3580 unmatched barangays...
INFO:modules.psgc_consolidator:Spatial matching complete: 2 direct matches, 3578 nearest neighbor fallbacks
INFO:modules.psgc_consolidator:Updating 3580 rows with spatially matched admin codes
INFO:modules.psgc_consolidator:Spatial matching results:
INFO:modules.psgc_consolidator:  - Spatially matched: 3580 barangays
INFO:modules.psgc_consolidator:  - Still unmatched: 0 barangays
INFO:modules.psgc_consolidator:  - Total features: 42048



AFTER SPATIAL MATCHING
Unmatched barangays: 0
Spatially matched barangays: 3580

DIAGNOSTIC: Reference Boundaries
Shape: (1582, 7)

Sample of reference boundaries:


,adm1_psgc,adm2_psgc,adm3_psgc,adm1_en,adm2_en,adm3_en,geometry
0,0100000000,0102800000,0102801000,Region I (Ilocos Region),Ilocos Norte,Adams,"POLYGON ((120.92068 18.51462, 120.94626 18.511..."
1,0100000000,0102800000,0102802000,Region I (Ilocos Region),Ilocos Norte,Bacarra,"POLYGON ((120.60543 18.2282, 120.60357 18.2288..."
2,0100000000,0102800000,0102803000,Region I (Ilocos Region),Ilocos Norte,Badoc,"POLYGON ((120.48513 17.89702, 120.48438 17.897..."
3,0100000000,0102800000,0102804000,Region I (Ilocos Region),Ilocos Norte,Bangui,"POLYGON ((120.70617 18.49936, 120.70553 18.500..."
4,0100000000,0102800000,0102805000,Region I (Ilocos Region),Ilocos Norte,City of Batac,"POLYGON ((120.55352 17.98857, 120.5534 17.9885..."



NaN counts in reference boundaries:
adm1_en    0
adm2_en    0
adm3_en    1
dtype: int64


In [4]:
# DIAGNOSTIC: Check source data PSGC code formats
print("\n" + "="*80)
print("DIAGNOSTIC: Source Data PSGC Code Formats")
print("="*80)

print("\nAdm1 data (Regions):")
print(f"  Shape: {consolidator_matched.adm1_data.shape}")
print(f"  adm1_psgc dtype: {consolidator_matched.adm1_data['adm1_psgc'].dtype}")
print(f"  Sample values: {consolidator_matched.adm1_data['adm1_psgc'].head(3).tolist()}")
print(f"  Sample after _add_leading_zeros: {[consolidator_matched._add_leading_zeros(x) for x in consolidator_matched.adm1_data['adm1_psgc'].head(3)]}")

print("\nAdm2 data (Provinces):")
print(f"  Shape: {consolidator_matched.adm2_data.shape}")
print(f"  adm2_psgc dtype: {consolidator_matched.adm2_data['adm2_psgc'].dtype}")
print(f"  Sample values: {consolidator_matched.adm2_data['adm2_psgc'].head(3).tolist()}")

print("\nAdm3 data (Municipalities):")
print(f"  Shape: {consolidator_matched.adm3_data.shape}")
print(f"  adm3_psgc dtype: {consolidator_matched.adm3_data['adm3_psgc'].dtype}")
print(f"  Sample values: {consolidator_matched.adm3_data['adm3_psgc'].head(3).tolist()}")

print("\nReference boundaries PSGC codes:")
print(f"  adm1_psgc dtype: {consolidator_matched.reference_boundaries['adm1_psgc'].dtype}")
print(f"  Sample values: {consolidator_matched.reference_boundaries['adm1_psgc'].head(3).tolist()}")


DIAGNOSTIC: Source Data PSGC Code Formats

Adm1 data (Regions):
  Shape: (17, 7)
  adm1_psgc dtype: int64
  Sample values: [100000000, 200000000, 300000000]
  Sample after _add_leading_zeros: ['0100000000', '0200000000', '0300000000']

Adm2 data (Provinces):
  Shape: (88, 8)
  adm2_psgc dtype: int64
  Sample values: [102800000, 102900000, 103300000]

Adm3 data (Municipalities):
  Shape: (1642, 9)
  adm3_psgc dtype: int64
  Sample values: [102801000, 102802000, 102803000]

Reference boundaries PSGC codes:
  adm1_psgc dtype: string
  Sample values: ['0100000000', '0100000000', '0100000000']


In [5]:
# Inspect spatially matched rows
spatially_matched = matched_gdf[matched_gdf['is_spatially_matched'] == True].copy()

print(f"Total spatially matched: {len(spatially_matched)}")
print("\nSample of spatially matched barangays:")
print("="*80)
display(spatially_matched[['psgc_code', 'adm4_en_psgc', 'adm1_en', 'adm2_en', 'adm3_en', 'is_spatially_matched']].head(10))

Total spatially matched: 3580

Sample of spatially matched barangays:


,psgc_code,adm4_en_psgc,adm1_en,adm2_en,adm3_en,is_spatially_matched
7236,0305401001,Agapito del Rosario,Region III (Central Luzon),Pampanga,Porac,True
7237,0305401002,Anunas,Region III (Central Luzon),Pampanga,Porac,True
7238,0305401003,Balibago,Region III (Central Luzon),Pampanga,Mabalacat City,True
7239,0305401004,Capaya,Region III (Central Luzon),Pampanga,Mexico,True
7240,0305401005,Claro M. Recto,Region III (Central Luzon),Pampanga,Porac,True
7241,0305401006,Cuayan,Region III (Central Luzon),Pampanga,Porac,True
7242,0305401007,Cutcut,Region III (Central Luzon),Pampanga,Porac,True
7243,0305401008,Cutud,Region III (Central Luzon),Pampanga,Mexico,True
7244,0305401010,Lourdes North West,Region III (Central Luzon),Pampanga,Porac,True
7245,0305401011,Lourdes Sur,Region III (Central Luzon),Pampanga,Porac,True


In [7]:
matched_gdf

,psgc_code,corr_code,name,adm4_en_psgc,geometry,adm1_psgc,adm2_psgc,adm3_psgc,adm4_psgc,adm1_en,adm2_en,adm3_en,adm4_en_shapes,geo_level,len_crs,area_crs,len_km,area_km2,is_spatially_matched
0,0102801001,12801001.0,Adams,Adams (Pob.),"POLYGON ((120.92068 18.51462, 120.94626 18.511...",0100000000,0102800000,0102801000,0102801001,Region I (Ilocos Region),Ilocos Norte,Adams,Adams,Bgy,45997.0,111184551.0,45.0,111.0,False
1,0102802001,12802001.0,Bani,Bani,"POLYGON ((120.61278 18.2759, 120.61282 18.2758...",0100000000,0102800000,0102802000,0102802001,Region I (Ilocos Region),Ilocos Norte,Bacarra,Bani,Bgy,5982.0,1761135.0,5.0,1.0,False
2,0102802002,12802002.0,Buyon,Buyon,"POLYGON ((120.62741 18.24638, 120.6337 18.2413...",0100000000,0102800000,0102802000,0102802002,Region I (Ilocos Region),Ilocos Norte,Bacarra,Buyon,Bgy,9117.0,3875134.0,9.0,3.0,False
3,0102802003,12802003.0,Cabaruan,Cabaruan,"POLYGON ((120.58982 18.26839, 120.60407 18.259...",0100000000,0102800000,0102802000,0102802003,Region I (Ilocos Region),Ilocos Norte,Bacarra,Cabaruan,Bgy,7745.0,2987648.0,7.0,2.0,False
4,0102802004,12802004.0,Cabulalaan,Cabulalaan,"POLYGON ((120.58368 18.2821, 120.58401 18.2820...",0100000000,0102800000,0102802000,0102802004,Region I (Ilocos Region),Ilocos Norte,Bacarra,Cabulalaan,Bgy,4502.0,1018354.0,4.0,1.0,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
42043,1909908004,NaN,None,Lagunde,"POLYGON ((124.6232 7.07579, 124.62318 7.07054,...",1200000000,1204700000,1204712000,<NA>,Region XII (SOCCSKSARGEN),Cotabato,Pikit,NaN,NaN,NaN,NaN,NaN,NaN,True
42044,1909908005,NaN,None,Macasendeg,"POLYGON ((124.56663 7.05121, 124.56074 7.0323,...",1200000000,1204700000,1204712000,<NA>,Region XII (SOCCSKSARGEN),Cotabato,Pikit,NaN,NaN,NaN,NaN,NaN,NaN,True
42045,1909908006,NaN,None,Manaulanan,"POLYGON ((124.62026 7.05817, 124.6205 7.05709,...",1200000000,1204700000,1204712000,<NA>,Region XII (SOCCSKSARGEN),Cotabato,Pikit,NaN,NaN,NaN,NaN,NaN,NaN,True
42046,1909908007,NaN,None,Pamalian,"POLYGON ((124.61168 7.05983, 124.61191 7.04989...",1200000000,1204700000,1204712000,<NA>,Region XII (SOCCSKSARGEN),Cotabato,Pikit,NaN,NaN,NaN,NaN,NaN,NaN,True


In [8]:
# Export both versions for comparison
output_dir = get_path('output')

# # Export original (with NaN rows)
# original_file = output_dir / 'consolidated_geodata_original.gpkg'
# consolidator_matched.export_original(str(original_file))
# print(f"✓ Exported original: {original_file}")

# Export matched (complete dataset with is_spatially_matched column)
matched_file = os.path.join(output_dir, 'consolidated_geodata_spatial_matched.gpkg')
consolidator_matched.export_matched(str(matched_file))
print(f"✓ Exported matched: {matched_file}")

INFO:pyogrio._io:Created 42,048 records
INFO:modules.psgc_consolidator:Exported 42048 matched features to GeoPackage: /workspace/project_paaral/output/consolidated_geodata_spatial_matched.gpkg


✓ Exported matched: /workspace/project_paaral/output/consolidated_geodata_spatial_matched.gpkg


In [ ]:
# Method 2: Automatic - Apply spatial matching during processing
# Uncomment to use this method:

# consolidator_auto = pc.PSGCConsolidator(base_path=path, verbose=True)
# consolidated_auto = consolidator_auto.process(auto_spatial_match=True)
# 
# print(f"Unmatched barangays: {consolidated_auto['adm1_psgc'].isna().sum()}")
# print(f"Spatially matched barangays: {consolidated_auto['is_spatially_matched'].sum()}")